# 한국어 챗봇 (polyglot-ko-5.8b + LoRA, A100)

A100(40GB) 기준. 4bit 없이 bf16 LoRA로 학습(품질↑). 데이터 3종 혼합.

런타임 → 런타임 유형 변경 → **A100 GPU**, 고RAM 권장.

In [ ]:
!pip -q install transformers datasets accelerate peft
!pip -q uninstall -y torchao

In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0))
print('bf16 지원:', torch.cuda.is_bf16_supported())

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT='/content/drive/MyDrive/polyglot58_a100_ckpt'

In [ ]:
from datasets import load_dataset, concatenate_datasets
def to_io(ds):
    def m(ex):
        instr=(ex.get('instruction') or '').strip()
        inp=(ex.get('input') or '').strip()
        return {'q': instr+('\n'+inp if inp else ''), 'a': str(ex.get('output') or '').strip()}
    return ds.map(m, remove_columns=ds.column_names)
parts=[]
for name,n in [('beomi/KoAlpaca-v1.1a',20000),('nlpai-lab/kullm-v2',25000),('kyujinpy/KOpen-platypus',12000)]:
    d=load_dataset(name, split='train'); d=d.select(range(min(n,len(d))))
    parts.append(to_io(d))
data=concatenate_datasets(parts).shuffle(seed=42)
data=data.filter(lambda e: len(e['a'])>5 and len(e['q'])>3)
print('총 학습 샘플:', len(data))

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
BASE='EleutherAI/polyglot-ko-5.8b'
tok=AutoTokenizer.from_pretrained(BASE); tok.pad_token=tok.eos_token
model=AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.bfloat16, device_map='auto')
model.gradient_checkpointing_enable(); model.enable_input_require_grads()
rt=tok.decode(tok.encode('파이썬이 뭐야?'),skip_special_tokens=True); assert '파이썬' in rt.replace(' ',''); print('토크나이저 OK')

In [ ]:
PROMPT='### 질문: {q}\n\n### 답변: {a}'
MAX_LEN=512
def fmt(ex):
    return tok(PROMPT.format(q=ex['q'],a=ex['a'])+tok.eos_token, truncation=True, max_length=MAX_LEN)
tokenized=data.map(fmt, remove_columns=data.column_names); print(tokenized)

In [ ]:
lora=LoraConfig(r=32, lora_alpha=64, lora_dropout=0.05, task_type='CAUSAL_LM',
    target_modules=['query_key_value','dense','dense_h_to_4h','dense_4h_to_h'])
model=get_peft_model(model, lora); model.print_trainable_parameters()

In [ ]:
import os
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling
collator=DataCollatorForLanguageModeling(tok, mlm=False)
args=TrainingArguments(output_dir=OUT, per_device_train_batch_size=4,
    gradient_accumulation_steps=4, num_train_epochs=3, learning_rate=2e-4,
    bf16=True, warmup_ratio=0.03, logging_steps=20, save_steps=300,
    save_total_limit=2, report_to='none')
resume=os.path.isdir(OUT) and any(p.startswith('checkpoint') for p in os.listdir(OUT))
Trainer(model=model,args=args,train_dataset=tokenized,data_collator=collator).train(resume_from_checkpoint=resume)

In [ ]:
# 어댑터를 본체에 병합 → 일반 모델로 저장 (서빙 시 peft 불필요)
model.gradient_checkpointing_disable()
merged=model.merge_and_unload()
SAVE='/content/polyglot58-chatbot'
merged.save_pretrained(SAVE, safe_serialization=True); tok.save_pretrained(SAVE)
def ask(q,n=150):
    p='### 질문: '+q+'\n\n### 답변:'
    ids=tok.encode(p,return_tensors='pt').to(merged.device)
    o=merged.generate(ids,max_new_tokens=n,do_sample=True,top_p=0.92,temperature=0.7,
        repetition_penalty=1.1,pad_token_id=tok.pad_token_id,eos_token_id=tok.eos_token_id)
    return tok.decode(o[0][ids.shape[1]:],skip_special_tokens=True).split('###')[0].strip()
for q in ['파이썬이 뭐야?','건강하게 사는 법 알려줘','좋은 개발자가 되려면?','오늘 기분이 안 좋아']:
    print('Q:',q); print('A:',ask(q)); print()

In [ ]:
from huggingface_hub import login, create_repo
login(token='hf_본인_WRITE_토큰')
REPO='jjminu/polyglot58-chatbot'
create_repo(REPO, exist_ok=True)
merged.push_to_hub(REPO); tok.push_to_hub(REPO)
print('업로드 완료:', REPO)